In [ ]:
import numpy as np
import yaml
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline  
plt.rcParams.update({'font.size': 12})
plt.rcParams['figure.figsize'] = [8, 4]

# Our model

Our baseline model is the defaults at 
https://lambda.gsfc.nasa.gov/toolbox/camb_online.html

I ran some single-parameter variations there, to create the derivative models below.
I'm using the "*_scalcls.dat" file from the camb output.

I'm going to pack those up in a single file, so you can read that instead of downloading all these files.

The parameters here are:

- The baryon physical density, $\omega_b \equiv \Omega_b h^2$
- The cold dark matter density, $\omega_c \equiv \Omega_c h^2$
- The Lambda density, $\Omega_\Lambda$
- The curvature density, $ \Omega_k$
   - (note that the derivatives of $\Omega_\Lambda$ and  $ \Omega_k$ are equal, if we hold all other things constant.)
- The Hubble constant, $H_0$
- Total matter density, $\Omega_m$
- 
  

In [ ]:
# This cell will only run if you have all the camb output files in the relevant places on your machine.
# You don't need to run it in order to do the in-class exercises.

# Create the derivative vectors, d(signal)/d(parameter)
# baseline model: 
#  Obh^2 = w_b = 0.0226
#  Och^2 = w_c = 0.112
#  H0 = 70


# Read in base model
ell, TTbase, _,_,_,_ = np.loadtxt('cambs/baseline/camb_79976740_scalcls.dat',unpack=True)
# find curvature derivative at fixed H0 and therefore fixed O_matter;  this is changing O_Lambda to change the curvature.

ell, TT, _,_,_,_  = np.loadtxt('cambs/Ok_plus_0p1/camb_13464442_scalcls.dat',unpack=True)
deriv_Ok = (TT-TTbase)/0.1  # d(D_ell)/d(Ok);  the 0.1 takes care of the actual change in the file.
deriv_OL = deriv_Ok  # at fixed HO, O_matter*h^2, this is true.

# Obh2 derivative
ell, TT, _,_,_,_ = np.loadtxt('cambs/Obh2_plus_0p01/camb_96545059_scalcls.dat',unpack=True)
deriv_Obh2 = (TT-TTbase)/0.01  #d(D_ell)/d(Obh2);  the 0.01 takes care of the actual change in the file.

# Och2 derivative
ell, TT, _,_,_,_ = np.loadtxt('cambs/Och2_plus_0p1/camb_33671803_scalcls.dat',unpack=True)
deriv_Och2 = (TT-TTbase)/0.1  #d(D_ell)/d(Och2);  the 0.1 takes care of the actual change in the file.

# H0 derivative
ell, TT, _,_,_,_ = np.loadtxt('cambs/H0_plus_5/camb_88429271_scalcls.dat',unpack=True)
deriv_H0 = (TT-TTbase)/5  #d(D_ell)/d(H0);  the 5 takes care of the actual change in the file.

# Om derivative
ell, TT, _,_,_,_ = np.loadtxt('cambs/Om_plus_0p1/camb_09205409_scalcls.dat',unpack=True)
deriv_Om = (TT-TTbase)/0.1  #d(D_ell)/d(Om);  the O.1 takes care of the actual change in the file.

# Save these to a text file - this is the one you'll use.
data = np.array([ell, TTbase, deriv_Ok, deriv_Obh2, deriv_Och2, deriv_H0, deriv_OL, deriv_Om])
np.savetxt('derivs.dat',data.T,fmt='%10.3e', header='#ell, TTbase, deriv_Ok, deriv_Obh2, deriv_Och2, deriv_H0, deriv_OL, deriv_Om')




In [ ]:
ell, TTbase, deriv_Ok, deriv_Obh2, deriv_Och2, deriv_H0, deriv_OL, deriv_Om = np.loadtxt('derivs.dat',unpack=True)

In [ ]:
plt.rcParams.update({'font.size': 12})
plt.rcParams['figure.figsize'] = [8, 4]

veclist = [deriv_Ok, deriv_Obh2, deriv_Och2, deriv_H0, deriv_OL] #, deriv_Om]
vecnames = ['deriv_Ok', 'deriv_Obh2', 'deriv_Och2', 'deriv_H0', 'deriv_OL', 'deriv_Om']
for ii in range(len(veclist)):
    plt.plot(ell,veclist[ii]/np.max(np.abs(veclist[ii])),label=vecnames[ii])

plt.legend()
plt.title('normalized derivative vectors')
plt.xlabel('$\ell$')
plt.grid()
plt.show()

### Question:
These look really random.  You can think of them as a set of basis vectors.  How would you figure out whether they are "orthogonal"?

# Make a CMB temperature anistropy "data" vector, with noise, and plot it.

Play with the "Nell" parameter, which sets the amount of instrument noise, to see how it affects the hash on the plot.

Then, pick a value between 0.005 and 0.02 to create the data vector you'll use for model fitting, below.

In [ ]:
# Initialize the random number generator
rng1 = np.random.default_rng()

# set up a "realistic" noise vs ell for an all-sky CMB experiment.
# If you set Nell=0, with no instrument noise, you'll still get "hash" due 
# to sample variance, ie we only have one CMB sky to look at, so only (2*ell+1) values of "m" to average over at each "ell".
#
Nell = 0.005
sigma = np.sqrt(2/(2*ell+1)) * (TTbase + ell*(ell+1)*Nell/(2*np.pi) )   # fancy equation
# Create the noise.
noise = rng1.normal(scale = sigma, size=len(ell))

# Data = Signal + noise
TTdata = TTbase + noise
plt.plot(ell,TTdata)
plt.grid()
plt.show()

# Single parameter fit

Let's leave all the parameters except one fixed to their default value.
We then vary the one parameter of interest, and plot the reduced chisquare vs that parameter value.
The ``best fit" value of that parameter is at the minimum of the chiquare.  (If the reduced chisquare 
is not very near 1 near its minimum, something is going wrong, often a mis-estimation of the errorbars.)

We'll also use the regular (not reduced) chisquare to find a one-sigma confidence interval for the parameter of interest.  That "1-sigma" error bound is where the chisquare increases by 1, so we can use that fact to find +- 1sigma range.

Our data are in the TTdata vector we set up above;  let's call those values $d_i$ ("d" for "data").  We know how much noise we added, so we're going to remember the amplitude of that, which is stored in the vector called "noise", and
we'll call those values $\sigma_i$.

The x-axis on that plot is Legendre polynomial value $\ell$, and our model makes a prediction for what those 
values should be;  let's call those model predictions $m_i$ ("m" for "model").

The chisquare statistic is given by 

$$ \chi^2 = \sum_i \frac{( d_i - m_i )^2}{\sigma_i^2} \ .$$

To find the ``reduced chisquare", we need to divide by the degrees of freedom, which is typically assumed to be

dof = $\nu$ = (number of data points) - (number of fit parameters)

Given that, the reduced chisquare is

$$ \chi^2_\nu = \frac{\chi^2}{\nu} .$$

Look for the "ALL CAPS" stuff below, which you'll need to play with.

In [ ]:
# Single parameter fit on Obh2

# Baseline model defaults in case we need them.
w_b0 = 0.0226
w_c0 = 0.112
H0 = 70

# PLAY WITH THIS RANGE TO ZOOM IN ON AREA OF INTEREST
# set parameter range:
wb_range = np.linspace(w_b0-0.05, w_b0+0.05,100)  

# Create a vector in which to stuff the chisquare results.
chsq_vec = np.array([])
degrees_of_freedom = len(ell) - 1  # number of data points, minus the one parameter we're fitting for.

for wb in wb_range:
    delta_wb = wb - w_b0
    TT = TTbase + delta_wb*deriv_Obh2

    #YOU WRITE A CHISQUARE FORMULA HERE.  # Don't use a library, it's easy math.
    #  Data vector: TTdata
    #  Model values: TT
    #  sigma values: sigma
    chisq = #YOU WRITE A CHISQUARE FORMULA HERE.
    #
    chsq_vec = np.append(chsq_vec,chisq)

# report minimum
ii = np.argmin(chsq_vec)
print(f'Minimum chisq = {chsq_vec[ii]:0.2f}, at w_b = {wb_range[ii]:0.5f}')

# report range over which delta_reduced_chisq < 1
delta_chisq = chsq_vec - np.min(chsq_vec)
jj = np.argwhere(delta_chisq<1)
wb_min = wb_range[jj[0]]
wb_max = wb_range[jj[-1]]
print(f'1 sigma range for wb is {wb_min[0]:0.5f} to {wb_max[0]:0.5f}')  

# PLOT THE REDUCED CHISQUARE
plt.subplot(2,1,1)
plt.plot(wb_range,chisq_vec/degrees_of_freedom)
plt.grid()
plt.ylim(0)

# PLOT THE CHISQUARE... (and zoom in on parameter range where it changes by 1
plt.subplot(2,1,2)
plt.plot(wb_range,chsq_vec)
plt.grid()
plt.ylim(0)


# Pick another parameter, and repeat this exercise below, ie a one-dimensional chisq curve.

In [ ]:
# Your code here, for another parameter.

# 2-parameter fit

Let's scan over two parameters and create a 2D grid of chisq vs them, 
and plot contours of the delta(chisq).  

We're going to do this by first creating 2D arrays of parameter velues to use for 
the cold dark matter and baryon densities.

In [ ]:
# First, let's set up the ranges, and make some 2D arrays of coordinates that are 
#  handy for using plt.contour().

# Play with the 0.1's here to change the range over which you're calculating.
wc_range = np.linspace(w_b0-0.1,w_b0+0.1,100)
wb_range = np.linspace(w_c0-0.1,w_c0+0.1,100)

wc, wb = np.meshgrid(wc_range,wb_range,indexing='ij') #

# Let's pause here and show what wc and wb look like
# Change this to wb and repeat.
# Look at how imshow orients things vs how contour does it.

plt.rcParams['figure.figsize'] = [10, 4]
plt.subplot(1,2,1)
plt.imshow(wc)
plt.title('wc')
plt.colorbar()
#
plt.subplot(1,2,2)
plt.imshow(wb)
plt.title('wb)
plt.colorbar()

#This will help you understand which index is which.
print(f'wb[0,0]= {wb[0,0]}')
print(f'wb[0,99]= {wb[0,99]}')
print(f'wb[99,0]= {wb[99,0]}')
print(f'wb[99,99]= {wb[99,99]}')

# Having learned about meshgrid and 2d array indexing, let's dive in.

In [ ]:
# Play with the 0.1's here to change the range over which you're calculating.
wc_range = np.linspace(w_b0-0.1,w_b0+0.1,100)
wb_range = np.linspace(w_c0-0.1,w_c0+0.1,100)

wc, wb = np.meshgrid(wc_range,wb_range,indexing='ij') #

chsq_array = np.zeros_like(wc)
wc_array = np.zeros_like(wc)    # if we're confident we don't have to use these, but it's good for debugging.
wb_array = np.zeros_like(wc)

# two loops
for bb in range(len(wb_range)):
    delta_wb = wb_range[bb] - w_b0
    for cc in range(len(wc_range)):
        delta_wc = wc_range[cc] - w_c0
        
        TT = TTbase + delta_wc*deriv_Och2 + delta_wb*deriv_Obh2

        # calculate the chisq.  Note, dof should be n_data_points -2 , for 2 fit parameters.
        chisq = YOUR FORMULA FOR CHISQUARE HERE

        # store everything.
        chsq_array[cc,bb] = chisq
        wc_array[cc,bb] = wc_range[cc]
        wb_array[cc,bb] = wb_range[bb]

# report minimum
ii = np.argmin(chsq_array.flatten())  # hint;  use this with wc_array.flatten()[]
wc_fit = wc_array.flatten()[ii]
wb_fit = wb_array.flatten()[ii]
print(f'Minimum chisq = {chsq_array.flatten()[ii]:0.2f}, at w_c = {wc_fit:0.5f}, w_b = {wb_fit:0.5f}')
plt.scatter(wb_fit,wc_fit)  # plot a point at the best-fit spot.

#plt.rcParams['figure.figsize'] = [4, 4]
#plt.contour(rchsq_array-np.min(rchsq_array),levels=[1,2,3,4,5])

dchisq = chsq_array - np.min(chsq_array)

plt.rcParams['figure.figsize'] = [5,5]
plt.contour(wb, wc, chisq,levels=[1,2,3,6,10,20])
plt.xlabel('w_b')
plt.ylabel('w_c')
plt.grid()

# Now repeat this for a different set of 2 parameters.

In [ ]:
# Your code here.

# If you have time:

Go to the lambda.gsfsc.nasa.gov website and change a different parameter (not in the set I gave you).
Use that new data file, along with a baseline file, to calculate the derivative spectrum for that parameters.
Plot it, then use it in making a 1D or 2D contour of delta_chisq.